# 可选实验：线性回归的梯度下降

<figure>
    <center> <img src="./images/C1_W1_L4_S1_Lecture_GD.png"  style="width:800px;height:200px;" ></center>
</figure>

## 目标
在本实验中，你将：
- 使用梯度下降自动执行优化 $w$ 和 $b$ 的过程。

## 工具
在本实验中，我们将使用：
- NumPy：常用的科学计算库；
- Matplotlib：常用的数据绘图库；
- 本地目录中 lab_utils.py 文件里的绘图例程。

In [1]:
import math, copy
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('./deeplearning.mplstyle')
from lab_utils_uni import plt_house_x, plt_contour_wgrad, plt_divergence, plt_gradients

<a name="toc_40291_2"></a>
# 问题陈述

让我们使用与之前相同的两个数据点——一套 1000 平方英尺的房屋售价为 \\$300,000，另一套 2000 平方英尺的房屋售价为 \\$500,000。

| 面积（1000 平方英尺）     | 价格（千美元） |
| ----------------| ------------------------ |
| 1               | 300                      |
| 2               | 500                      |

In [ ]:
# Load our data set
x_train = np.array([1.0, 2.0])   #features
y_train = np.array([300.0, 500.0])   #target value

<a name="toc_40291_2.0.1"></a>
### Compute_Cost
这是在上一个实验中开发的。此处还会再次用到它。

In [ ]:
#Function to calculate the cost
def compute_cost(x, y, w, b):
   
    m = x.shape[0] 
    cost = 0
    
    for i in range(m):
        f_wb = w * x[i] + b
        cost = cost + (f_wb - y[i])**2
    total_cost = 1 / (2 * m) * cost

    return total_cost

<a name="toc_40291_2.1"></a>
## 梯度下降总结
到目前为止，你已经构建了一个用于预测 $f_{w,b}(x^{(i)})$ 的线性模型：
$$f_{w,b}(x^{(i)}) = wx^{(i)} + b \tag{1}$$
在线性回归中，你利用输入训练数据来拟合参数 $w$、$b$，方法是最小化预测值 $f_{w,b}(x^{(i)})$ 与实际数据 $y^{(i)}$ 之间的误差度量。这个度量称为$cost$，即 $J(w,b)$。在训练中，需要衡量所有训练样本 $x^{(i)},y^{(i)}$ 上的代价
$$J(w,b) = \frac{1}{2m} \sum\limits_{i = 0}^{m-1} (f_{w,b}(x^{(i)}) - y^{(i)})^2\tag{2}$$ 


课程中将*梯度下降*描述为：

$$\begin{align*} \text{repeat}&\text{ until convergence:} \; \lbrace \newline
\;  w &= w -  \alpha \frac{\partial J(w,b)}{\partial w} \tag{3}  \; \newline 
 b &= b -  \alpha \frac{\partial J(w,b)}{\partial b}  \newline \rbrace
\end{align*}$$
其中，参数 $w$、$b$ 同时更新。  
梯度定义为：
$$
\begin{align}
\frac{\partial J(w,b)}{\partial w}  &= \frac{1}{m} \sum\limits_{i = 0}^{m-1} (f_{w,b}(x^{(i)}) - y^{(i)})x^{(i)} \tag{4}\\
  \frac{\partial J(w,b)}{\partial b}  &= \frac{1}{m} \sum\limits_{i = 0}^{m-1} (f_{w,b}(x^{(i)}) - y^{(i)}) \tag{5}\\
\end{align}
$$

这里的*同时*意味着先计算所有参数的偏导数，然后再更新任何一个参数。

<a name="toc_40291_2.2"></a>
## 实现梯度下降
您将为单个特征实现梯度下降算法，需要三个函数：
- `compute_gradient`：实现上面的公式 (4) 和 (5)；
- `compute_cost`：实现上面的公式 (2)（代码来自上一个实验）；
- `gradient_descent`：使用 compute_gradient 和 compute_cost。

约定：
- Python 中表示偏导数的变量命名遵循以下模式：$\frac{\partial J(w,b)}{\partial b}$ 将写作 `dj_db`；
- w.r.t 表示“相对于”（With Respect To），例如 $J(wb)$ 相对于 $b$ 的偏导数。

<a name="toc_40291_2.3"></a>
### compute_gradient
<a name='ex-01'></a>
`compute_gradient` 实现了上面的公式 (4) 和 (5)，并返回 $\frac{\partial J(w,b)}{\partial w}$、$\frac{\partial J(w,b)}{\partial b}$。其中的注释说明了各项操作。

In [ ]:
def compute_gradient(x, y, w, b): 
    """
    Computes the gradient for linear regression 
    Args:
      x (ndarray (m,)): Data, m examples 
      y (ndarray (m,)): target values
      w,b (scalar)    : model parameters  
    Returns
      dj_dw (scalar): The gradient of the cost w.r.t. the parameters w
      dj_db (scalar): The gradient of the cost w.r.t. the parameter b     
     """
    
    # Number of training examples
    m = x.shape[0]    
    dj_dw = 0
    dj_db = 0
    
    for i in range(m):  
        f_wb = w * x[i] + b 
        dj_dw_i = (f_wb - y[i]) * x[i] 
        dj_db_i = f_wb - y[i] 
        dj_db += dj_db_i
        dj_dw += dj_dw_i 
    dj_dw = dj_dw / m 
    dj_db = dj_db / m 
        
    return dj_dw, dj_db

<br/>

<img align="left" src="./images/C1_W1_Lab03_lecture_slopes.PNG"   style="width:340px;" > 课程介绍了梯度下降如何利用某一点处代价对参数的偏导数来更新该参数。  
让我们使用 `compute_gradient` 函数，找出并绘制代价函数相对于其中一个参数 $w_0$ 的若干偏导数。

In [ ]:
plt_gradients(x_train,y_train, compute_cost, compute_gradient)
plt.show()

上面的左图显示了三个点处的 $\frac{\partial J(w,b)}{\partial w}$，即代价曲线相对于 $w$ 的斜率。在图的右侧，导数为正；在左侧，导数为负。由于其“碗状”形态，导数总会引导梯度下降走向底部，也就是梯度为零的位置。
 
左图固定了 $b=100$。梯度下降将同时利用 $\frac{\partial J(w,b)}{\partial w}$ 和 $\frac{\partial J(w,b)}{\partial b}$ 更新参数。右侧的“箭袋图”提供了一种查看两个参数梯度的方式。箭头大小反映该点处梯度的幅值，箭头的方向和斜率反映该点处 $\frac{\partial J(w,b)}{\partial w}$ 与 $\frac{\partial J(w,b)}{\partial b}$ 的比值。
请注意，梯度指向*远离*最小值的方向。回顾上面的方程 (3)。当前的 $w$ 或 $b$ 会*减去*经过缩放的梯度，从而使参数朝着降低代价的方向移动。

<a name="toc_40291_2.5"></a>
### 梯度下降
现在已经可以计算梯度，接下来便可以在 `gradient_descent` 中实现上面方程 (3) 所描述的梯度下降。实现细节已在注释中说明。下面，你将使用此函数在训练数据上寻找 $w$ 和 $b$ 的最优值。

In [ ]:
def gradient_descent(x, y, w_in, b_in, alpha, num_iters, cost_function, gradient_function): 
    """
    Performs gradient descent to fit w,b. Updates w,b by taking 
    num_iters gradient steps with learning rate alpha
    
    Args:
      x (ndarray (m,))  : Data, m examples 
      y (ndarray (m,))  : target values
      w_in,b_in (scalar): initial values of model parameters  
      alpha (float):     Learning rate
      num_iters (int):   number of iterations to run gradient descent
      cost_function:     function to call to produce cost
      gradient_function: function to call to produce gradient
      
    Returns:
      w (scalar): Updated value of parameter after running gradient descent
      b (scalar): Updated value of parameter after running gradient descent
      J_history (List): History of cost values
      p_history (list): History of parameters [w,b] 
      """
    
    w = copy.deepcopy(w_in) # avoid modifying global w_in
    # An array to store cost J and w's at each iteration primarily for graphing later
    J_history = []
    p_history = []
    b = b_in
    w = w_in
    
    for i in range(num_iters):
        # Calculate the gradient and update the parameters using gradient_function
        dj_dw, dj_db = gradient_function(x, y, w , b)     

        # Update Parameters using equation (3) above
        b = b - alpha * dj_db                            
        w = w - alpha * dj_dw                            

        # Save cost J at each iteration
        if i<100000:      # prevent resource exhaustion 
            J_history.append( cost_function(x, y, w , b))
            p_history.append([w,b])
        # Print cost every at intervals 10 times or as many iterations if < 10
        if i% math.ceil(num_iters/10) == 0:
            print(f"Iteration {i:4}: Cost {J_history[-1]:0.2e} ",
                  f"dj_dw: {dj_dw: 0.3e}, dj_db: {dj_db: 0.3e}  ",
                  f"w: {w: 0.3e}, b:{b: 0.5e}")
 
    return w, b, J_history, p_history #return w and J,w history for graphing

In [ ]:
# initialize parameters
w_init = 0
b_init = 0
# some gradient descent settings
iterations = 10000
tmp_alpha = 1.0e-2
# run gradient descent
w_final, b_final, J_hist, p_hist = gradient_descent(x_train ,y_train, w_init, b_init, tmp_alpha, 
                                                    iterations, compute_cost, compute_gradient)
print(f"(w,b) found by gradient descent: ({w_final:8.4f},{b_final:8.4f})")

<img align="left" src="./images/C1_W1_Lab03_lecture_learningrate.PNG"  style="width:340px; padding: 15px; " >
花一点时间观察上面打印的梯度下降过程的一些特征。

- 如课程幻灯片所述，代价从较大值开始并迅速下降。
- 偏导数 `dj_dw` 和 `dj_db` 也在减小，开始时很快，之后逐渐变慢。如课程图示所示，当过程接近“碗底”时，由于该点处导数值较小，前进速度会减慢。
- 尽管学习率 alpha 保持不变，进展仍会变慢

### 代价与梯度下降迭代次数的关系
代价随迭代次数变化的图是衡量梯度下降进展的有效指标。在成功运行时，代价应始终减小。代价在初始阶段变化得非常快，因此使用不同于最终下降阶段的尺度绘制初始下降过程会很有帮助。在下图中，请注意坐标轴上的代价尺度和迭代步长。

In [ ]:
# plot cost versus iteration  
fig, (ax1, ax2) = plt.subplots(1, 2, constrained_layout=True, figsize=(12,4))
ax1.plot(J_hist[:100])
ax2.plot(1000 + np.arange(len(J_hist[1000:])), J_hist[1000:])
ax1.set_title("Cost vs. iteration(start)");  ax2.set_title("Cost vs. iteration (end)")
ax1.set_ylabel('Cost')            ;  ax2.set_ylabel('Cost') 
ax1.set_xlabel('iteration step')  ;  ax2.set_xlabel('iteration step') 
plt.show()

### 预测
现在你已经找到了参数 $w$ 和 $b$ 的最优值，可以使用模型根据学到的参数预测房价。正如预期的那样，对于相同的房屋，预测值与训练值几乎一致。此外，未包含在训练数据中的预测值也符合预期。

In [ ]:
print(f"1000 sqft house prediction {w_final*1.0 + b_final:0.1f} Thousand dollars")
print(f"1200 sqft house prediction {w_final*1.2 + b_final:0.1f} Thousand dollars")
print(f"2000 sqft house prediction {w_final*2.0 + b_final:0.1f} Thousand dollars")

<a name="toc_40291_2.6"></a>
## 绘图
你可以在代价 cost(w,b) 的等高线图上绘制代价随迭代次数的变化，从而展示梯度下降执行期间的进展。

In [ ]:
fig, ax = plt.subplots(1,1, figsize=(12, 6))
plt_contour_wgrad(x_train, y_train, p_hist, ax)

上面的等高线图显示了 $cost(w,b)$ 在 $w$ 和 $b$ 一定范围内的值。各个环表示不同的代价水平。图中叠加的红色箭头表示梯度下降的路径。需要注意以下几点：
- 该路径持续（单调）向目标前进。
- 最初的步长远大于接近目标时的步长。

**放大观察**，可以看到梯度下降的最后几个步骤。请注意，随着梯度接近零，各步之间的距离不断缩小。

In [ ]:
fig, ax = plt.subplots(1,1, figsize=(12, 4))
plt_contour_wgrad(x_train, y_train, p_hist, ax, w_range=[180, 220, 0.5], b_range=[80, 120, 0.5],
            contours=[1,5,10,20],resolution=0.5)

<a name="toc_40291_2.7.1"></a>
### 提高学习率

<figure>
 <img align="left", src="./images/C1_W1_Lab03_alpha_too_big.PNG"   style="width:340px;height:240px;" >
</figure>
课程中讨论过公式 (3) 里学习率 $\alpha$ 的合适取值。$\alpha$ 越大，梯度下降收敛到解的速度越快。但是，如果它过大，梯度下降就会发散。上面展示了一个顺利收敛的解。

让我们尝试增大 $\alpha$ 的值，看看会发生什么：

In [ ]:
# initialize parameters
w_init = 0
b_init = 0
# set alpha to a large value
iterations = 10
tmp_alpha = 8.0e-1
# run gradient descent
w_final, b_final, J_hist, p_hist = gradient_descent(x_train ,y_train, w_init, b_init, tmp_alpha, 
                                                    iterations, compute_cost, compute_gradient)

上面，$w$ 和 $b$ 在正负值之间来回振荡，而且绝对值随每次迭代而增大。此外，每次迭代中 $\frac{\partial J(w,b)}{\partial w}$ 都会改变符号，并且代价不降反升。这是*学习率过大*、解正在发散的明显迹象。
让我们用图形将其可视化。

In [ ]:
plt_divergence(p_hist, J_hist,x_train, y_train)
plt.show()

在上图中，左图显示了梯度下降最初几个步骤中 $w$ 的变化过程。$w$ 在正值和负值之间振荡，代价迅速增大。梯度下降会同时作用于 $w$ 和 $b$，因此需要右侧的三维图才能看清全貌。


## 恭喜！
在本实验中，你：
- 深入研究了单变量梯度下降的细节。
- 开发了一个计算梯度的例程
- 将梯度可视化
- 完成了一个梯度下降例程
- 利用梯度下降寻找参数
- 检查了学习率大小的影响